# Validação de sistemas de RAG

## 1) Carregamento de bibliotecas

In [15]:
import warnings

warnings.filterwarnings('ignore')

from langchain_community.document_loaders import DirectoryLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_openai import ChatOpenAI
from langchain_openai.embeddings import OpenAIEmbeddings
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_classic.retrievers import MultiQueryRetriever
from langchain_core.output_parsers import StrOutputParser, CommaSeparatedListOutputParser, JsonOutputParser
from langchain_classic.evaluation.qa import QAGenerateChain
from langchain_classic.evaluation.qa import QAEvalChain
from langchain_community.vectorstores import FAISS
from transformers import AutoTokenizer
from langchain_openai.embeddings import OpenAIEmbeddings
from operator import itemgetter
from dotenv import load_dotenv
import httpx
import os

import json

http_client = httpx.Client(verify=False)

In [2]:
os.chdir(r'c:\Users\francisco.bneto\Documents\gen-ai-formation')
print(os.getcwd())

c:\Users\francisco.bneto\Documents\gen-ai-formation


In [3]:
_ = load_dotenv()
api_key = os.getenv('OPENROUTER_API_KEY')
api_url = os.getenv('OPENROUTER_BASE_URL')
print(f"Variáveis de ambiente carregadas: {os.getenv('OPENROUTER_API_KEY')[:10]}...")

Variáveis de ambiente carregadas: sk-or-v1-c...


In [6]:
# definição de modelos
embedding_model = OpenAIEmbeddings(
    model="openai/text-embedding-3-small",
    openai_api_key=api_key,
    openai_api_base=api_url,
    http_client=http_client
)

llm_model = ChatOpenAI(
    model="meta-llama/llama-3.1-8b-instruct",
    api_key=api_key,
    base_url=api_url,
    http_client=http_client
)

eval_chain = QAEvalChain.from_llm(llm_model)

In [39]:
def avaliar(perguntas_respostas, geracoes):
    # perguntas_respostas: query, answer
    # geracoes: result
    avaliacoes = eval_chain.evaluate(perguntas_respostas, geracoes)
    corretas = 0
    for i, _ in enumerate(perguntas_respostas):
        corretas += (1 if avaliacoes[i]["results"].split("\n")[-1].split(":")[-1].strip() == "CORRECT" else 0)

    return corretas / len(perguntas_respostas)

## 2) Criação de chunks

In [5]:
# Carregamento de dados
print("Carregamento dos dados...")
pdfs = DirectoryLoader("./documentos", glob="*.pdf").load()

# Criação de chunks
print("Criação dos chunks...")
tokenizer = AutoTokenizer.from_pretrained("BAAI/bge-m3")
splitter = CharacterTextSplitter.from_huggingface_tokenizer(
    tokenizer=tokenizer,
    chunk_size=1250,
    chunk_overlap=150
)

chunks = splitter.split_documents(pdfs)

# Criação do banco vetorial
print("Criação do banco vetorial...")
vector_store = FAISS.from_documents(
    documents=chunks,
    embedding=embedding_model
)

Carregamento dos dados...
Criação dos chunks...


Criação do banco vetorial...


## 3) Construção da chain de avaliação

In [8]:
prompt_generation = PromptTemplate.from_template(
"""
Com base no texto abaixo, gere uma pergunta e resposta em JSON.
Retorne APENAS o JSON no formato: {{"question": "...", "answer": "..."}}

Não retorne nada além do JSON solicitado.
Texto: {doc}
"""
)

qa_chain = prompt_generation | llm_model | JsonOutputParser()

perguntas_respostas = [
    qa_chain.invoke({"doc": p.page_content}) for p in chunks
]

In [14]:
perguntas_respostas

[{'question': 'Como um portador de cartão Mastercard Gold deve emitir os Bilhetes de Seguro para obter a cobertura da Garantia Estendida Original? ',
  'answer': 'através do portal www.aig.com/Mastercard/pt.'},
 {'question': 'Quem está coberto?',
  'answer': 'Os portadores de cartão Mastercard Gold.'},
 {'question': 'Qual é a lista dos Bens Elegíveis a este Seguro?',
  'answer': "A lista dos Bens Elegíveis a este Seguro inclui: Antena Parabólica, Áudio Portátil, Áudio System, Auto Rádio, DVD Player, Karaokê, Videokê, Blue ray, GPS, Home theater com ou sem dvd e blu-ray, Filmadora Digital, Lente para máquina fotográfica e para celulares, Máquina fotográfica digital, MP3 Player, MP4 Player, MP5 Player, iPod, Dock Station, Receptor / Decodificador / Conversor de sinal digital, Telão de Projeção ou Datashow, Televisor Convencional / LCD / LED, Televisor de Plasma, Vídeo Game, Ar Condicionado Janela / Split / Portátil, Bebedouro de Água Elétrico ou Purificador de Água Elétrico, Coifa, Depur

In [16]:
with open("qa_pairs.json", "w", encoding="utf-8") as file:
    json.dump(perguntas_respostas, file, ensure_ascii=False, indent=4)

In [17]:
perguntas_respostas[0]

{'question': 'Como um portador de cartão Mastercard Gold deve emitir os Bilhetes de Seguro para obter a cobertura da Garantia Estendida Original? ',
 'answer': 'através do portal www.aig.com/Mastercard/pt.'}

In [18]:
geracoes_sem_rag = list()
for pr in perguntas_respostas[:10]:
    geracoes_sem_rag.append({"result": llm_model.invoke(pr["question"])})

In [25]:
geracoes_sem_rag = [{"result": g['result'].content} for g in geracoes_sem_rag]

In [26]:
geracoes_sem_rag

[{'result': 'Como portador de um cartão Mastercard Gold, para emitir os Bilhetes de Seguro (BS) e obter a cobertura da Garantia Estendida Original (GEO), você pode seguir os passos abaixo:\n\n1. **Verifique se seu cartão tem a GEO**: Certifique-se de que seu cartão Mastercard Gold tenha a Garantia Estendida Original ativada. Isso geralmente é indicado no contrato de cartão ou na página do site do banco que emitiu o cartão.\n2. **Entre no site do banco**: Acesse o site do banco que emitiu seu cartão Mastercard Gold e entre na seção de cartões de crédito ou seguros.\n3. **Faça o login**: Faça o login com suas credenciais (nome de usuário e senha) para acessar a área de clientela.\n4. **Clique em "Seguro" ou "Bilhete de Seguro"**: Procure a opção de seguros ou bilhetes de seguro e clique nela para entrar na área específica.\n5. **Escolha a opção de emitir BS**: Escolha a opção de emitir Bilhetes de Seguro, que geralmente é uma opção disponível no menu de seguros.\n6. **Preencha as informa

In [ ]:
perguntas_respostas = [{**{k:v for k, v in p.items() if k != "question"}, "query": p["question"]} for p in perguntas_respostas]

KeyError: 'question'

In [37]:
perguntas_respostas[0]

{'answer': 'através do portal www.aig.com/Mastercard/pt.',
 'query': 'Como um portador de cartão Mastercard Gold deve emitir os Bilhetes de Seguro para obter a cobertura da Garantia Estendida Original? '}

In [40]:
avaliar(perguntas_respostas[:10], geracoes_sem_rag)

0.0

## Avaliação do RAG chain

In [41]:
retriever = vector_store.as_retriever()

prompt_template = """
Você é um especialista em questões bancárias, especialmente em dúvidas relacionadas a cartões de crédito.

Responsa as perguntas usando exclusivamente os conteúdos fornecidos.

Contexto:
{contexto}
"""

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", prompt_template),
        ("human", "{query}")
    ]
)

rag_chain = (
    {
        "contexto": itemgetter("query") | retriever,
        "query": itemgetter("query")
    }
    | prompt | llm_model | StrOutputParser()
)

In [42]:
geracoes_rag = []

for pr in perguntas_respostas[:10]:
    geracoes_rag.append(
        {
            "result": rag_chain.invoke(
                {
                    "query": pr["query"]
                }
            )
        }
    )

In [43]:
avaliar(perguntas_respostas[:10], geracoes_rag)

0.3